# Extraction des Topics : Que disent les clients ?

## Contexte
On a un modèle capable de classifier les avis en positif/négatif/neutre.
Maintenant on veut aller plus loin : **de quoi parlent les clients ?**

On va utiliser deux approches complémentaires :
1. **TF-IDF** → les mots les plus importants par sentiment
2. **LDA** → les thèmes cohérents qui ressortent des avis

## Ce notebook
1. Charger les données nettoyées
2. Extraire les mots clés par sentiment (TF-IDF)
3. Extraire les thèmes principaux (LDA)
4. Interpréter les résultats

## 1. Chargement des librairies

In [1]:
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer, CountVectorizer
from sklearn.decomposition import LatentDirichletAllocation
import numpy as np

print(" Librairies chargées avec succès ")

 Librairies chargées avec succès 


## 2. Chargement des données

In [ ]:
# Charger les données nettoyées
df = pd.read_csv("../outputs/02_data_nettoyee.csv")

print(f"Nombre d'avis : {len(df)}")
print(f"\nRépartition des sentiments :")
for sentiment, nombre in df["sentiment"].value_counts().items():
    pourcentage = (nombre / len(df)) * 100
    print(f"  {sentiment:>8} : {nombre:>6} avis ({pourcentage:.1f}%)")

# Séparer les avis par sentiment
positifs = df[df["sentiment"] == "positif"]["text_nettoye"]
negatifs = df[df["sentiment"] == "negatif"]["text_nettoye"]
neutres  = df[df["sentiment"] == "neutre"]["text_nettoye"]

print(f"\n Données séparées par sentiment")

Nombre d'avis : 568453

Répartition des sentiments :
   positif : 443776 avis (78.1%)
   negatif :  82037 avis (14.4%)
    neutre :  42640 avis (7.5%)

 Données séparées par sentiment !


## 3. Mots clés par sentiment (TF-IDF)

On utilise TF-IDF pour trouver les mots les plus importants
dans chaque groupe d'avis (positif, négatif, neutre).

Un mot avec un score TF-IDF élevé dans les avis négatifs
signifie qu'il est très représentatif des plaintes clients.

In [3]:
def extraire_mots_cles(textes, n_mots=20):
    """
    Extrait les n mots les plus importants d'un groupe d'avis.
    
    Paramètres :
    - textes : liste d'avis nettoyés
    - n_mots : nombre de mots clés à extraire
    
    Retourne :
    - liste des mots clés avec leur score TF-IDF moyen
    """
    vectorizer = TfidfVectorizer(max_features=5000)
    tfidf_matrix = vectorizer.fit_transform(textes)
    
    # Score moyen de chaque mot sur tous les avis
    scores_moyens = tfidf_matrix.mean(axis=0).A1
    
    # Récupérer les mots et leurs scores
    mots = vectorizer.get_feature_names_out()
    
    # Trier par score décroissant
    indices_tries = scores_moyens.argsort()[::-1][:n_mots]
    
    return [(mots[i], round(scores_moyens[i], 4)) for i in indices_tries]

# Extraire les mots clés pour chaque sentiment
print("=== TOP 20 mots clés POSITIFS ===")
for mot, score in extraire_mots_cles(positifs):
    print(f"  {mot:<20} : {score}")

print("\n=== TOP 20 mots clés NÉGATIFS ===")
for mot, score in extraire_mots_cles(negatifs):
    print(f"  {mot:<20} : {score}")

print("\n=== TOP 20 mots clés NEUTRES ===")
for mot, score in extraire_mots_cles(neutres):
    print(f"  {mot:<20} : {score}")

=== TOP 20 mots clés POSITIFS ===
  great                : 0.0292
  coffee               : 0.0286
  love                 : 0.0285
  like                 : 0.0279
  good                 : 0.0277
  tea                  : 0.0261
  taste                : 0.0252
  flavor               : 0.0241
  product              : 0.0237
  br                   : 0.0227
  one                  : 0.0224
  food                 : 0.0193
  dog                  : 0.0185
  get                  : 0.0165
  best                 : 0.0162
  make                 : 0.016
  price                : 0.0158
  really               : 0.0157
  would                : 0.015
  time                 : 0.0149

=== TOP 20 mots clés NÉGATIFS ===
  like                 : 0.0324
  taste                : 0.0321
  product              : 0.0297
  coffee               : 0.0263
  br                   : 0.0256
  one                  : 0.0231
  flavor               : 0.0228
  would                : 0.0217
  tea                  : 0.0204
  goo

### Problème détecté

On voir que beaucoup de mots apparaissent dans les 3 sentiments. On va donc affiner l'extraction en gardant uniquement les mots **exclusifs** à chaque sentiment ie ceux qui n'apparaissent pas (ou peu) dans les autres classes. Au lieu de prendre les mots les plus fréquents dans chaque classe, on prend les mots qui apparaissent beaucoup plus dans une classe que dans les autres.

On remarque aussi le mot `br` (balise HTML) à supprimer.

In [4]:
# Supprimer le mot "br" (balise HTML résiduelle)
df["text_nettoye"] = df["text_nettoye"].str.replace(r'\bbr\b', '', regex=True)

# Recréer les groupes
positifs = df[df["sentiment"] == "positif"]["text_nettoye"]
negatifs = df[df["sentiment"] == "negatif"]["text_nettoye"]
neutres  = df[df["sentiment"] == "neutre"]["text_nettoye"]

def extraire_mots_exclusifs(textes_cible, textes_autres, n_mots=15):
    """
    Extrait les mots les plus représentatifs d'une classe
    en comparant avec les autres classes.
    
    Paramètres :
    - textes_cible : avis de la classe qu'on analyse
    - textes_autres : avis de toutes les autres classes
    - n_mots : nombre de mots à extraire
    """
    # TF-IDF sur la classe cible
    vec_cible = TfidfVectorizer(max_features=5000)
    score_cible = vec_cible.fit_transform(textes_cible).mean(axis=0).A1
    mots_cible = dict(zip(vec_cible.get_feature_names_out(), score_cible))
    
    # TF-IDF sur les autres classes
    vec_autres = TfidfVectorizer(max_features=5000, vocabulary=vec_cible.vocabulary_)
    score_autres = vec_autres.fit_transform(textes_autres).mean(axis=0).A1
    mots_autres = dict(zip(vec_autres.get_feature_names_out(), score_autres))
    
    # Score exclusif = score dans la cible - score dans les autres
    scores_exclusifs = {
        mot: mots_cible[mot] - mots_autres.get(mot, 0)
        for mot in mots_cible
    }
    
    # Trier par score exclusif décroissant
    top_mots = sorted(scores_exclusifs.items(),
                     key=lambda x: x[1],
                     reverse=True)[:n_mots]
    
    return top_mots

# Combiner les avis non-positifs, non-négatifs, non-neutres
autres_positifs = df[df["sentiment"] != "positif"]["text_nettoye"]
autres_negatifs = df[df["sentiment"] != "negatif"]["text_nettoye"]
autres_neutres  = df[df["sentiment"] != "neutre"]["text_nettoye"]

print("===  TOP 15 mots EXCLUSIFS POSITIFS ===")
for mot, score in extraire_mots_exclusifs(positifs, autres_positifs):
    print(f"  {mot:<20} : {round(score, 4)}")

print("\n===  TOP 15 mots EXCLUSIFS NÉGATIFS ===")
for mot, score in extraire_mots_exclusifs(negatifs, autres_negatifs):
    print(f"  {mot:<20} : {round(score, 4)}")

print("\n===  TOP 15 mots EXCLUSIFS NEUTRES ===")
for mot, score in extraire_mots_exclusifs(neutres, autres_neutres):
    print(f"  {mot:<20} : {round(score, 4)}")

===  TOP 15 mots EXCLUSIFS POSITIFS ===
  great                : 0.017
  love                 : 0.015
  best                 : 0.01
  delicious            : 0.0077
  perfect              : 0.0069
  favorite             : 0.0066
  snack                : 0.0057
  highly               : 0.0054
  easy                 : 0.0051
  excellent            : 0.0051
  find                 : 0.0049
  wonderful            : 0.0049
  good                 : 0.0048
  tea                  : 0.0045
  make                 : 0.0042

===  TOP 15 mots EXCLUSIFS NÉGATIFS ===
  disappointed         : 0.0075
  money                : 0.007
  bad                  : 0.0068
  box                  : 0.0065
  taste                : 0.006
  would                : 0.0057
  waste                : 0.0057
  product              : 0.0056
  thought              : 0.0053
  return               : 0.0051
  review               : 0.0049
  didnt                : 0.0049
  awful                : 0.0048
  away                 : 0.00

### Résultats de l'extraction TF-IDF

####  Avis positifs
Mots dominants : `great`, `love`, `best`, `delicious`, `perfect`. Les clients parlent de **qualité**, **goût** et **satisfaction**

####  Avis négatifs
Mots dominants : `disappointed`, `waste`, `return`, `awful`, `horrible`. Les clients parlent de **déception**, **argent perdu** et **retours produit**

####  Avis neutres
Mots dominants : `ok`, `however`, `probably`, `bit`. Les clients utilisent des mots de **nuance** et d'**hésitation**

### Conclusion
L'extraction TF-IDF donne des mots clés très cohérents.
On passe maintenant au LDA pour trouver des thèmes cohérents.

## 4. Extraction des thèmes principaux (LDA)

Le LDA (Latent Dirichlet Allocation) est un algorithme probabiliste qui regroupe automatiquement les mots en thèmes.

On applique le LDA séparément sur :
- Les avis positifs pour identifier quels thèmes satisfont les clients ?
- Les avis négatifs pour identifier quels thèmes mécontentent les clients ?

Paramètres choisis :
- K=5 topics : bon équilibre détail/lisibilité
- 10 mots par topic pour l'interprétation

In [5]:
def extraire_topics_lda(textes, n_topics=5, n_mots=10):
    """
    Extrait les topics principaux d'un groupe d'avis avec LDA.
    
    Paramètres :
    - textes    : liste d'avis nettoyés
    - n_topics  : nombre de topics à extraire
    - n_mots    : nombre de mots par topic
    """
    # CountVectorizer pour le LDA (fréquences brutes, pas TF-IDF)
    vectorizer = CountVectorizer(
        max_features=5000,
        min_df=5,      # mot présent dans au moins 5 avis
        max_df=0.95    # mot présent dans moins de 95% des avis
    )
    
    matrice = vectorizer.fit_transform(textes)
    mots    = vectorizer.get_feature_names_out()
    
    # Entraîner le modèle LDA
    lda = LatentDirichletAllocation(
        n_components=n_topics,
        random_state=42,
        max_iter=10
    )
    lda.fit(matrice)
    
    # Extraire les mots principaux de chaque topic
    topics = []
    for i, topic in enumerate(lda.components_):
        top_indices = topic.argsort()[::-1][:n_mots]
        top_mots    = [mots[j] for j in top_indices]
        topics.append(top_mots)
    
    return topics

# Appliquer sur les avis positifs
print("=== TOPICS POSITIFS ===\n")
topics_positifs = extraire_topics_lda(positifs)
for i, mots in enumerate(topics_positifs):
    print(f"Topic {i+1} : {', '.join(mots)}")

# Appliquer sur les avis négatifs
print("\n=== TOPICS NÉGATIFS ===\n")
topics_negatifs = extraire_topics_lda(negatifs)
for i, mots in enumerate(topics_negatifs):
    print(f"Topic {i+1} : {', '.join(mots)}")

=== TOPICS POSITIFS ===

Topic 1 : coffee, cup, like, flavor, taste, good, one, great, strong, blend
Topic 2 : tea, taste, like, flavor, drink, good, one, love, great, water
Topic 3 : food, dog, treat, love, cat, one, like, get, eat, product
Topic 4 : like, taste, flavor, good, great, chocolate, love, snack, one, chip
Topic 5 : product, amazon, price, great, store, good, find, time, one, buy

=== TOPICS NÉGATIFS ===

Topic 1 : dog, food, cat, treat, product, one, would, eat, like, made
Topic 2 : coffee, taste, like, cup, flavor, one, good, chocolate, would, tried
Topic 3 : like, taste, flavor, one, good, would, product, chip, really, dont
Topic 4 : product, amazon, box, would, one, bag, order, item, price, time
Topic 5 : tea, taste, like, product, flavor, sugar, water, ingredient, drink, one


### Résultats LDA

#### Topics positifs
| Topic | Thème | Mots clés |
|---|---|---|
| 1 | ☕ Café | coffee, cup, flavor, strong, blend |
| 2 | 🍵 Thé | tea, drink, love, water |
| 3 | 🐾 Animaux | dog, treat, cat, eat |
| 4 | 🍫 Snacks | chocolate, snack, chip |
| 5 | 💰 Prix/Achat | amazon, price, store, buy |

#### Topics négatifs
| Topic | Thème | Mots clés |
|---|---|---|
| 1 | 🐾 Animaux refusent | dog, cat, eat, would |
| 2 | ☕ Déception café | coffee, flavor, tried |
| 3 | 🍫 Snacks décevants | chip, dont, really |
| 4 | 📦 Livraison | amazon, box, order, item |
| 5 | 🧪 Ingrédients | sugar, ingredient, water |

### Conclusion
Le LDA révèle des thèmes très cohérents et actionnables.
L'entreprise peut utiliser ces insights pour :
- Améliorer la livraison et le packaging (Topic négatif 4)
- Mettre en avant le goût du café et du thé (Topics positifs 1 et 2)
- Revoir les recettes des snacks (Topic négatif 3)

## 5. Sauvegarde des résultats

On sauvegarde les mots clés et les topics dans des fichiers CSV pour les réutiliser dans le dashboard Streamlit.

In [6]:
# Sauvegarder les mots clés TF-IDF
mots_positifs = extraire_mots_exclusifs(positifs, autres_positifs)
mots_negatifs = extraire_mots_exclusifs(negatifs, autres_negatifs)
mots_neutres  = extraire_mots_exclusifs(neutres, autres_neutres)

df_mots_cles = pd.DataFrame({
    "mot_positif"   : [m for m, s in mots_positifs],
    "score_positif" : [s for m, s in mots_positifs],
    "mot_negatif"   : [m for m, s in mots_negatifs],
    "score_negatif" : [s for m, s in mots_negatifs],
    "mot_neutre"    : [m for m, s in mots_neutres],
    "score_neutre"  : [s for m, s in mots_neutres],
})
df_mots_cles.to_csv("../outputs/04_mots_cles.csv", index=False)

# Sauvegarder les topics LDA
rows_positifs = [{"sentiment": "positif",
                  "topic": f"Topic {i+1}",
                  "mots": ", ".join(mots)}
                 for i, mots in enumerate(topics_positifs)]

rows_negatifs = [{"sentiment": "negatif",
                  "topic": f"Topic {i+1}",
                  "mots": ", ".join(mots)}
                 for i, mots in enumerate(topics_negatifs)]

df_topics = pd.DataFrame(rows_positifs + rows_negatifs)
df_topics.to_csv("../outputs/04_topics_lda.csv", index=False)

print(" Mots clés sauvegardés dans outputs/04_mots_cles.csv")
print(" Topics LDA sauvegardés dans outputs/04_topics_lda.csv")

 Mots clés sauvegardés dans outputs/04_mots_cles.csv
 Topics LDA sauvegardés dans outputs/04_topics_lda.csv


## Résumé de ce notebook

Dans ce notebook on a :

1. **Chargé** les données nettoyées (568 453 avis)
2. **Extrait** les mots clés exclusifs par sentiment (TF-IDF)
3. **Corrigé** un résidu HTML (`br`) détecté dans les données
4. **Extrait** les thèmes principaux par sentiment (LDA, K=5)
5. **Sauvegardé** les résultats dans `outputs/`

### Insights clés pour l'entreprise

#### Ce que les clients aiment ✅
- Le goût du café et du thé
- Les snacks et chocolats
- Les prix et la disponibilité sur Amazon

#### Ce que les clients n'aiment pas ❌
- Les problèmes de livraison et packaging
- Les ingrédients de certains produits
- Les snacks qui ne correspondent pas aux attentes